In [20]:
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
from sklearn.naive_bayes import MultinomialNB
import numpy as np

# 1. Caricamento semplice (senza pulizia aggressiva per il naive)
#    Il naive non deve essere "aiutato" troppo.
naive_train_df = pd.read_csv("development.csv")
# Creiamo la funzione per non ripetere il codice
def preprocess_df(df):
    return df['title'].fillna('') + " " + df['article'].fillna('')
X = preprocess_df(naive_train_df)
y = naive_train_df['label']

# Split semplice per valutazione interna
X_train_n, X_val_n, y_train_n, y_val_n = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- BASELINE 2: NAIVE BAYES (Minimal Intelligence) ---
# Questo è il confronto più onesto per un modello di NLP.
naive_pipe = Pipeline([
    ('vect', TfidfVectorizer()), # Nessun parametro custom, tutto default
    ('clf', MultinomialNB())     # Modello probabilistico semplice
])

naive_pipe.fit(X_train_n, y_train_n)
naive_pred = naive_pipe.predict(X_val_n)
naive_f1 = f1_score(y_val_n, naive_pred, average='macro')

print(f"Naive Bayes F1-Macro: {naive_f1:.4f}")

Naive Bayes F1-Macro: 0.4626


In [21]:
# --- PREDIZIONE SU EVALUATION (Il Fix) ---
evaluation_df = pd.read_csv("evaluation.csv")
X_eval = preprocess_df(evaluation_df) # Applichiamo la stessa logica!

y_pred = naive_pipe.predict(X_eval)

# --- SUBMISSION SICURA ---
# Usiamo l'Id originale se presente nel CSV, altrimenti l'indice
submission = pd.DataFrame({
    'Id': evaluation_df.index, # O evaluation_df['Id'] se la colonna esiste
    'Predicted': y_pred
})
submission.to_csv("submission.csv", index=False)